# Module 8 — Lab 2: End-to-End RAG System Using LangChain
## Build a Complete Retrieval-Augmented Generation Pipeline with Chroma & FAISS Vector Stores

---

### Lab Overview

In production GenAI systems, LLMs often need to answer questions grounded in **private or domain-specific knowledge** — policy documents, research papers, internal wikis, medical records, or financial reports. Retrieval-Augmented Generation (RAG) solves this by fetching relevant document chunks at query time and injecting them as context into the LLM prompt.

In this lab, we build a **complete end-to-end RAG system** from scratch — loading documents (JSON + PDF), chunking them, embedding and indexing into vector stores (both **Chroma** and **FAISS**), performing semantic retrieval with different strategies (similarity search and MMR), and finally assembling a question-answering pipeline using **LangChain Expression Language (LCEL)**.

| Step | What We Do | Key Technology |
|------|-----------|----------------|
| 1 | Install dependencies & configure API keys | `langchain`, `langchain-openai`, `langchain-chroma`, `faiss-cpu` |
| 2 | Load documents from JSON (Wikipedia) and PDFs (research papers) | `JSONLoader`, `PyMuPDFLoader` |
| 3 | Chunk documents using recursive text splitting | `RecursiveCharacterTextSplitter` |
| 4 | Embed & index chunks in **Chroma** vector store | `OpenAIEmbeddings`, `Chroma` |
| 5 | Embed & index chunks in **FAISS** vector store | `FAISS` |
| 6 | Retrieve documents using similarity search and MMR | `as_retriever()` with different `search_type` |
| 7 | Build a RAG QA chain using LCEL pipe operator | `ChatPromptTemplate`, `RunnablePassthrough`, `ChatOpenAI` |
| 8 | Test the pipeline with domain-specific queries | End-to-end question answering |

### Learning Objectives

By the end of this lab, you will be able to:
1. Load and process heterogeneous document sources (JSON, PDF) using LangChain document loaders
2. Implement recursive character text splitting with configurable chunk size and overlap
3. Create and persist Chroma and FAISS vector stores with OpenAI embeddings
4. Compare similarity search vs. Maximum Marginal Relevance (MMR) retrieval strategies
5. Build a complete RAG pipeline using LangChain Expression Language (LCEL)
6. Test the system with diverse queries and understand retrieval grounding

---

## 1. Install Dependencies

We install the core LangChain packages, vector store integrations (Chroma + FAISS), the PDF parser (`pymupdf`), and the `jq` library for JSON document processing.

In [ ]:
# Core LangChain + OpenAI integration
!pip install langchain langchain-openai langchain-community langchain-chroma -q

# FAISS for vector similarity search (CPU version for Colab)
!pip install faiss-cpu -q

# PDF parsing (PyMuPDF) and JSON processing
!pip install pymupdf jq -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4

## 2. API Key Configuration

We load the OpenAI API key from **Google Colab's Secrets Manager**.

> **Setup:** Go to 🔑 icon in Colab's left sidebar → Add `OPENAI_API_KEY` → Toggle "Notebook access" ON.

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Verify key is loaded (never print the actual key)
status = "✅ Loaded" if os.environ.get("OPENAI_API_KEY") else "❌ MISSING"
print(f"OPENAI_API_KEY: {status}")

OPENAI_API_KEY: ✅ Loaded


## 3. Initialize the Embedding Model

OpenAI's `text-embedding-3-small` model generates 1536-dimensional vectors — a good balance of quality and cost for RAG applications. This is the model that will convert both our document chunks and user queries into vectors for similarity comparison.

| Model | Dimensions | Cost (per 1M tokens) | Best For |
|-------|-----------|----------------------|----------|
| `text-embedding-3-small` | 1536 | ~$0.02 | Most RAG use cases, cost-effective |
| `text-embedding-3-large` | 3072 | ~$0.13 | High-precision retrieval, large corpora |

In [ ]:
from langchain_openai import OpenAIEmbeddings

# Initialize the embedding model — used for both indexing and querying
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')
print(f"Embedding model: {openai_embed_model.model}")

Embedding model: text-embedding-3-small


## 4. Loading and Processing Documents

A real-world RAG system ingests documents from multiple sources. In this lab, we work with two document types:

- **JSON Lines (JSONL):** Wikipedia articles — structured text already organized into paragraphs
- **PDF files:** Research papers — unstructured, multi-page documents that need chunking

> **Business Analogy:** Think of this as a bank's knowledge management system ingesting both structured data (policy manuals in JSON from internal systems) and unstructured data (regulatory PDFs from RBI/SEBI circulars).

### 4.1 Download the Dataset

We download a pre-packaged dataset containing Wikipedia articles (JSONL) and AI/ML research papers (PDF).

In [ ]:
# Download the dataset ZIP from Google Drive
# If gdown fails, download manually from: https://drive.google.com/file/d/1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
!pip install gdown -q
!gdown 1aZxZejfteVuofISodUrY2CDoyuPLYDGZ

Downloading...
From: https://drive.google.com/uc?id=1aZxZejfteVuofISodUrY2CDoyuPLYDGZ
To: /content/rag_docs.zip
100% 5.92M/5.92M [00:00<00:00, 119MB/s]


In [ ]:
# Extract the documents into ./rag_docs/
!unzip -o rag_docs.zip

# Check what files we have
import os
for f in sorted(os.listdir('./rag_docs')):
    size_kb = os.path.getsize(f'./rag_docs/{f}') / 1024
    print(f"  {f:50s} ({size_kb:.1f} KB)")

Archive:  rag_docs.zip
   creating: rag_docs/
  inflating: rag_docs/attention_paper.pdf  
  inflating: rag_docs/cnn_paper.pdf  
  inflating: rag_docs/resnet_paper.pdf  
  inflating: rag_docs/vision_transformer.pdf  
  inflating: rag_docs/wikidata_rag_demo.jsonl  
  attention_paper.pdf                                (2163.3 KB)
  cnn_paper.pdf                                      (222.8 KB)
  resnet_paper.pdf                                   (800.2 KB)
  vision_transformer.pdf                             (3656.1 KB)
  wikidata_rag_demo.jsonl                            (1715.0 KB)


### 4.2 Load and Process JSON Documents (Wikipedia Articles)

The JSONL file contains Wikipedia articles with fields like `title`, `id`, and `paragraphs`. We use LangChain's `JSONLoader` to parse each line, then restructure the documents so that all paragraphs are combined into a single `page_content` string per article, with metadata preserved.

In [ ]:
from langchain_community.document_loaders import JSONLoader

# Load the JSONL file — each line is a separate JSON object
loader = JSONLoader(
    file_path='./rag_docs/wikidata_rag_demo.jsonl',
    jq_schema='.',           # parse each line as a complete JSON object
    text_content=False,       # content is JSON, not plain text
    json_lines=True           # file has one JSON object per line
)
wiki_docs = loader.load()

print(f"Loaded {len(wiki_docs)} Wikipedia articles")
print(f"Sample document keys: {list(wiki_docs[0].metadata.keys())}")

Loaded 1801 Wikipedia articles
Sample document keys: ['source', 'seq_num']


In [ ]:
import json
#from langchain.docstore.document import Document
from langchain_core.documents import Document

def process_wiki_documents(raw_docs):
    """Convert raw JSON documents into LangChain Document objects.

    Each Wikipedia article's paragraphs are joined into a single page_content,
    with title, id, and source preserved as metadata.
    """
    processed = []
    for doc in raw_docs:
        data = json.loads(doc.page_content)
        metadata = {
            "title": data['title'],
            "id": data['id'],
            "source": "Wikipedia"
        }
        # Combine all paragraphs into one text block
        content = ' '.join(data['paragraphs'])
        processed.append(Document(page_content=content, metadata=metadata))
    return processed

wiki_docs_processed = process_wiki_documents(wiki_docs)
print(f"Processed {len(wiki_docs_processed)} Wikipedia documents")

Processed 1801 Wikipedia documents


In [ ]:
wiki_docs[0]


Document(metadata={'source': '/content/rag_docs/wikidata_rag_demo.jsonl', 'seq_num': 1}, page_content='{"id": "84801", "title": "Chinese New Year", "paragraphs": ["Chinese New Year, known in China as the SpringFestival and in Singapore as the LunarNewYear, is a holiday on and around the new moon on the first day of the year in the traditional Chinese calendar. This calendar is based on the changes in the moon and is only sometimes changed to fit the seasons of the year based on how the Earth moves around the sun. Because of this, Chinese New Year is never on January1. It moves around between January21 and February20.", "The Chinese New Year is of the most important holidays for Chinese people all over the world. Its 7th day used to be used instead of birthdays to count people\'s ages in China. The holiday is still used to tell people which \\"animal\\" of the Chinese zodiac they are part of. The holiday is a time for gifts to children and for family gatherings with large meals, just li

In [ ]:
# Inspect a sample document — see its metadata and content preview
sample = wiki_docs_processed[3]
print(f"Title   : {sample.metadata['title']}")
print(f"Source  : {sample.metadata['source']}")
print(f"Length  : {len(sample.page_content):,} characters")
print(f"Preview : {sample.page_content[:300]}...")

Title   : Chi-square distribution
Source  : Wikipedia
Length  : 715 characters
Preview : In probability theory and statistics, the chi-square distribution (also chi-squared or formula_1  distribution) is one of the most widely used theoretical probability distributions. Chi-square distribution with formula_2 degrees of freedom is written as formula_3. It is a special case of gamma distr...


In [ ]:
sample

Document(metadata={'title': 'Chi-square distribution', 'id': '71548', 'source': 'Wikipedia'}, page_content='In probability theory and statistics, the chi-square distribution (also chi-squared or formula_1\xa0 distribution) is one of the most widely used theoretical probability distributions. Chi-square distribution with formula_2 degrees of freedom is written as formula_3. It is a special case of gamma distribution. Chi-square distribution is primarily used in statistical significance tests and confidence intervals. It is useful, because it is relatively easy to show that certain probability distributions come close to it, under certain conditions. One of these conditions is that the null hypothesis must be true. Another one is that the different random variables (or observations) must be independent of each other.')

### 4.3 Load and Chunk PDF Documents

PDF documents (research papers) are typically long and need to be split into smaller chunks for effective retrieval. We use:
- **`PyMuPDFLoader`** to extract text from PDF pages
- **`RecursiveCharacterTextSplitter`** to split the extracted text into chunks of ~3500 characters with 200-character overlap

> **Why chunk overlap?** Overlap ensures that if a relevant passage sits at the boundary of two chunks, it appears in both — preventing information loss during retrieval.

> **Business Analogy:** When an insurance company indexes policy documents, a 50-page policy PDF must be chunked so that a claims adjuster's query about "flood damage exclusion clause" retrieves just the relevant section, not the entire document.

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
#from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_simple_chunks(file_path, chunk_size=3500, chunk_overlap=200):
    """Load a PDF and split it into fixed-size text chunks.

    Args:
        file_path: Path to the PDF file
        chunk_size: Maximum characters per chunk (default 3500)
        chunk_overlap: Character overlap between consecutive chunks (default 200)

    Returns:
        List of Document objects (one per chunk)
    """
    print(f'Loading: {file_path}')
    loader = PyMuPDFLoader(file_path)
    doc_pages = loader.load()
    print(f'  → {len(doc_pages)} pages extracted')

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    doc_chunks = splitter.split_documents(doc_pages)
    print(f'  → {len(doc_chunks)} chunks created')
    return doc_chunks

In [ ]:
from glob import glob

# Find all PDF files in the dataset
pdf_files = sorted(glob('./rag_docs/*.pdf'))
print(f"Found {len(pdf_files)} PDF files:\n")

# Process each PDF into chunks
paper_docs = []
for fp in pdf_files:
    paper_docs.extend(create_simple_chunks(file_path=fp,
                                           chunk_size=3500,
                                           chunk_overlap=200))
    print()

print(f"\nTotal PDF chunks: {len(paper_docs)}")

Found 4 PDF files:

Loading: ./rag_docs/attention_paper.pdf
  → 15 pages extracted
  → 16 chunks created

Loading: ./rag_docs/cnn_paper.pdf
  → 11 pages extracted
  → 11 chunks created

Loading: ./rag_docs/resnet_paper.pdf
  → 12 pages extracted
  → 24 chunks created

Loading: ./rag_docs/vision_transformer.pdf
  → 22 pages extracted
  → 28 chunks created


Total PDF chunks: 79


In [ ]:
# Inspect the first chunk — see what metadata PyMuPDF extracts
print("=== First Chunk ===")
print(f"Metadata : {paper_docs[0].metadata}")
print(f"Length   : {len(paper_docs[0].page_content):,} characters")
print(f"Preview  : {paper_docs[0].page_content[:400]}...")

print("\n=== Last Chunk ===")
print(f"Length   : {len(paper_docs[-1].page_content):,} characters")

=== First Chunk ===
Metadata : {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'source': './rag_docs/attention_paper.pdf', 'file_path': './rag_docs/attention_paper.pdf', 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'trapped': '', 'modDate': 'D:20230803000729Z', 'creationDate': 'D:20230803000729Z', 'page': 0}
Length   : 2,857 characters
Preview  : Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
G...

=== Last Chunk ===
Length   : 625 characters


### 4.4 Combine All Document Chunks

We merge the Wikipedia articles and PDF chunks into a single unified corpus. This is the complete set of documents that will be indexed in our vector stores.

In [ ]:
# Combine Wikipedia articles + PDF chunks into one corpus
total_docs = wiki_docs_processed + paper_docs

print(f"Wikipedia articles : {len(wiki_docs_processed)}")
print(f"PDF chunks         : {len(paper_docs)}")
print(f"Total corpus size  : {len(total_docs)} documents")

# Quick stats on content lengths
lengths = [len(d.page_content) for d in total_docs]
print(f"\nContent length stats:")
print(f"  Min    : {min(lengths):,} chars")
print(f"  Max    : {max(lengths):,} chars")
print(f"  Avg    : {sum(lengths)//len(lengths):,} chars")

Wikipedia articles : 1801
PDF chunks         : 79
Total corpus size  : 1880 documents

Content length stats:
  Min    : 88 chars
  Max    : 7,395 chars
  Avg    : 960 chars


## 5. Indexing Documents in Chroma Vector Store

**Chroma** is a lightweight, open-source vector database that supports both in-memory and on-disk persistence. It's ideal for prototyping and small-to-medium RAG applications.

The indexing process:
1. Each document chunk is passed through the embedding model → 1536-dim vector
2. The vector + original text + metadata are stored in Chroma's collection
3. We configure **cosine similarity** as the distance function (Chroma defaults to L2/Euclidean)

> **Business Scenario:** A healthcare company indexes 500+ clinical guidelines and drug information leaflets. Doctors can then query: *"What are the contraindications for metformin in patients with renal impairment?"* and get the exact relevant paragraph.

In [ ]:
from langchain_chroma import Chroma

# Create Chroma vector DB — embeds all documents and stores to disk
# This takes ~30-60 seconds depending on corpus size
chroma_db = Chroma.from_documents(
    documents=total_docs,
    collection_name='rag_lab_collection',
    embedding=openai_embed_model,
    # Set cosine similarity (default is L2 Euclidean)
    # See: https://docs.trychroma.com/guides#changing-the-distance-function
    collection_metadata={"hnsw:space": "cosine"},
    persist_directory="./chroma_db"
)

print(f"✅ Chroma DB created with {chroma_db._collection.count()} documents")
print(f"   Persisted to: ./chroma_db/")

✅ Chroma DB created with 1880 documents
   Persisted to: ./chroma_db/


In [ ]:
import sqlite3
import pandas as pd

# 1. Connect to the Chroma SQLite database
# Update the path if your persist_directory is different
db_path = "./chroma_db/chroma.sqlite3"
conn = sqlite3.connect(db_path)

# 2. See what tables exist inside Chroma
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables_df = pd.read_sql_query(tables_query, conn)
print("Tables in ChromaDB:")
display(tables_df)

# 3. View the actual documents and metadata
# Chroma usually stores these in a table called 'embeddings' or 'embedding_metadata'
# Let's pull the first 5 rows from the embeddings table
try:
    data_df = pd.read_sql_query("SELECT * FROM embeddings LIMIT 5;", conn)
    print("\nSample Data from 'embeddings' table:")
    display(data_df)
except Exception as e:
    print(f"\nCouldn't query embeddings table: {e}")

# Always close the connection when you're done!
conn.close()

Tables in ChromaDB:


,name
0,migrations
1,acquire_write
2,collection_metadata
3,segment_metadata
4,tenants
5,databases
6,collections
7,maintenance_log
8,segments
9,embeddings



Sample Data from 'embeddings' table:


,id,segment_id,embedding_id,seq_id,created_at
0,1,01a0dbe3-c160-40d5-9a6d-3bc1603d04e5,8be83b19-7dbf-423f-8934-2895e164a088,1,2026-03-10 14:35:55
1,2,01a0dbe3-c160-40d5-9a6d-3bc1603d04e5,37a6cbdc-9af8-4d26-8528-cabc4d77e015,2,2026-03-10 14:35:55
2,3,01a0dbe3-c160-40d5-9a6d-3bc1603d04e5,dc09d880-c80f-4414-8a21-5a34cf7f70d8,3,2026-03-10 14:35:55
3,4,01a0dbe3-c160-40d5-9a6d-3bc1603d04e5,cc06b684-6657-41ae-b03e-3da7a6da839c,4,2026-03-10 14:35:55
4,5,01a0dbe3-c160-40d5-9a6d-3bc1603d04e5,540849a2-c5d7-4929-8dd9-1c8975a7adae,5,2026-03-10 14:35:55


### 5.1 Loading a Persisted Chroma DB from Disk

Once indexed, the vector store is saved to disk. In production, you'd build the index once (offline) and load it at application startup — avoiding re-embedding costs.

> **Tip:** This is equivalent to loading a pre-built search index in an enterprise system — you don't re-index every time the application restarts.

In [ ]:
# Load the persisted Chroma DB from disk (simulating a fresh session)
chroma_db = Chroma(
    persist_directory="./chroma_db",
    collection_name='rag_lab_collection',
    embedding_function=openai_embed_model
)

print(f"✅ Loaded Chroma DB from disk: {chroma_db._collection.count()} documents")

✅ Loaded Chroma DB from disk: 1880 documents


## 6. Indexing Documents in FAISS Vector Store

**FAISS** (Facebook AI Similarity Search) is Meta's high-performance library for dense vector similarity search. While Chroma is a full database with metadata filtering, FAISS is a pure vector index — faster for large-scale similarity search.

| Feature | Chroma | FAISS |
|---------|--------|-------|
| **Type** | Vector database | Vector index library |
| **Metadata filtering** | ✅ Built-in | ❌ Manual |
| **Persistence** | Built-in | Save/load index files |
| **Scalability** | Small-medium | Up to billions of vectors |
| **Best for** | Prototyping, metadata-rich queries | High-performance retrieval, large corpora |

> **Business Scenario:** An e-commerce platform like Flipkart indexes 10 million product descriptions for semantic search. FAISS handles this scale efficiently with approximate nearest neighbor (ANN) algorithms.

In [ ]:
from langchain_community.vectorstores import FAISS

# Create FAISS index from the same document corpus
faiss_db = FAISS.from_documents(
    documents=total_docs,
    embedding=openai_embed_model
)

print(f"✅ FAISS index created with {faiss_db.index.ntotal} vectors")
print(f"   Vector dimension: {faiss_db.index.d}")

✅ FAISS index created with 1880 vectors
   Vector dimension: 1536


In [ ]:
# Save FAISS index to disk
faiss_db.save_local("./faiss_index")
print("✅ FAISS index saved to ./faiss_index/")

# Load it back (simulating production startup)
faiss_db = FAISS.load_local(
    "./faiss_index",
    openai_embed_model,
    allow_dangerous_deserialization=True  # required for loading pickle files
)
print(f"✅ FAISS index loaded: {faiss_db.index.ntotal} vectors")

✅ FAISS index saved to ./faiss_index/
✅ FAISS index loaded: 1880 vectors


## 7. Retrieval Strategies — Similarity Search vs. MMR

A retriever converts a user query into an embedding and finds the most relevant document chunks. LangChain supports multiple retrieval strategies:

| Strategy | How It Works | When to Use |
|----------|-------------|-------------|
| **Similarity Search** | Returns top-K chunks closest to the query vector (cosine similarity) | Default choice; works well for focused queries |
| **MMR (Maximum Marginal Relevance)** | Balances relevance with diversity — avoids returning near-duplicate chunks | When documents have overlapping content; improves answer coverage |

> **Business Scenario:** When a banking customer asks *"What are the eligibility criteria for a home loan?"*, similarity search might return 5 chunks that all say the same thing (from overlapping sections). MMR would return diverse chunks covering income requirements, CIBIL score, property valuation, and documentation — giving a more complete answer.

In [ ]:
# --- Similarity Search Retriever (Chroma) ---
# Returns the top-5 most similar chunks based on cosine distance
similarity_retriever = chroma_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# --- MMR Retriever (Chroma) ---
# Balances relevance with diversity; fetch_k candidates → select k diverse ones
mmr_retriever = chroma_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,          # number of final results
        "fetch_k": 20    # candidates to consider before diversity filtering
    }
)

# --- FAISS Similarity Retriever ---
faiss_retriever = faiss_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("✅ Three retrievers initialized:")
print("   1. Chroma — Similarity Search (top-5)")
print("   2. Chroma — MMR (top-5 from 20 candidates)")
print("   3. FAISS  — Similarity Search (top-5)")

✅ Three retrievers initialized:
   1. Chroma — Similarity Search (top-5)
   2. Chroma — MMR (top-5 from 20 candidates)
   3. FAISS  — Similarity Search (top-5)


In [ ]:
from IPython.display import display, Markdown

def display_retrieved_docs(docs, label="Retrieved"):
    """Display retrieved document chunks with metadata and content preview."""
    print(f"\n{'='*70}")
    print(f"  {label} — {len(docs)} chunks")
    print(f"{'='*70}")
    for i, doc in enumerate(docs, 1):
        title = doc.metadata.get('title', doc.metadata.get('source', 'N/A'))
        print(f"\n--- Chunk {i} | Source: {title} ---")
        print(f"Content preview ({len(doc.page_content):,} chars):")
        display(Markdown(doc.page_content[:500] + "..."))

### 7.1 Test Similarity Search Retrieval

In [ ]:
# Query: a focused technical question
query = "What is machine learning and how does it relate to artificial intelligence?"
top_docs = similarity_retriever.invoke(query)
display_retrieved_docs(top_docs, label="Similarity Search (Chroma)")


  Similarity Search (Chroma) — 5 chunks

--- Chunk 1 | Source: Artificial intelligence ---
Content preview (1,218 chars):


Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is also a field of study which tries to make computers "smart". They work on their own without being encoded with commands. John McCarthy came up with the name "Artificial Intelligence" in 1955. In general use, the term "artificial intelligence" means a programme which mimics human cognition. At least some of the things we associate with other minds, such as learning and problem solving can be d...


--- Chunk 2 | Source: Machine learning ---
Content preview (743 chars):


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designin...


--- Chunk 3 | Source: Artificial neural network ---
Content preview (979 chars):


A neural network (also called an ANN or an artificial neural network) is a sort of computer software, inspired by biological neurons. Biological brains are capable of solving difficult problems, but each neuron is only responsible for solving a very small part of the problem. Similarly, a neural network is made up of cells that work together to produce a desired result, although each individual cell is only responsible for solving a small part of the problem. This is one method for creating arti...


--- Chunk 4 | Source: Deep learning ---
Content preview (1,053 chars):


Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks. As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised. In many cases, structures are organised so that there is at least one intermediate layer (or hidden layer), between the input layer and the output layer. Certain tasks, such as as recognizing and understanding speech, i...


--- Chunk 5 | Source: Supervised learning ---
Content preview (439 chars):


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data....

### 7.2 Test MMR Retrieval

Notice how MMR returns more **diverse** chunks compared to plain similarity search — reducing redundancy in the retrieved context.

In [ ]:
# Same query, but with MMR — expect more diverse results
query = "What is machine learning and how does it relate to artificial intelligence?"
mmr_docs = mmr_retriever.invoke(query)
display_retrieved_docs(mmr_docs, label="MMR Retrieval (Chroma)")


  MMR Retrieval (Chroma) — 5 chunks

--- Chunk 1 | Source: Artificial intelligence ---
Content preview (1,218 chars):


Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is also a field of study which tries to make computers "smart". They work on their own without being encoded with commands. John McCarthy came up with the name "Artificial Intelligence" in 1955. In general use, the term "artificial intelligence" means a programme which mimics human cognition. At least some of the things we associate with other minds, such as learning and problem solving can be d...


--- Chunk 2 | Source: Machine learning ---
Content preview (743 chars):


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designin...


--- Chunk 3 | Source: Supervised learning ---
Content preview (439 chars):


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data....


--- Chunk 4 | Source:  ---
Content preview (2,217 chars):


An Introduction to Convolutional Neural Networks
Keiron O’Shea1 and Ryan Nash2
1 Department of Computer Science, Aberystwyth University, Ceredigion, SY23 3DB
keo7@aber.ac.uk
2 School of Computing and Communications, Lancaster University, Lancashire, LA1
4YW
nashrd@live.lancs.ac.uk
Abstract. The ﬁeld of machine learning has taken a dramatic twist in re-
cent times, with the rise of the Artiﬁcial Neural Network (ANN). These
biologically inspired computational models are able to far exceed the per-...


--- Chunk 5 | Source: Loop AI Labs ---
Content preview (1,032 chars):


Loop AI Labs is an AI and cognitive computing company that focuses on language understanding technology. The company was founded in San Francisco in 2012 by Italian entrepreneur Gianmauro Calafiore, who sold his company Gsmbox to in 2004 and then relocated from Italy to San Francisco. Wanting to start an artificial intelligence company, he recruited two veterans of the project, the largest government-funded AI project in history, who had worked on the project at and Stanford University's . The o...

### 7.3 Test FAISS Retrieval

In [ ]:
# Same query on FAISS index
query = "What is machine learning and how does it relate to artificial intelligence?"
faiss_docs = faiss_retriever.invoke(query)
display_retrieved_docs(faiss_docs, label="Similarity Search (FAISS)")


  Similarity Search (FAISS) — 5 chunks

--- Chunk 1 | Source: Artificial intelligence ---
Content preview (1,218 chars):


Artificial intelligence (AI) is the ability of a computer program or a machine to think and learn. It is also a field of study which tries to make computers "smart". They work on their own without being encoded with commands. John McCarthy came up with the name "Artificial Intelligence" in 1955. In general use, the term "artificial intelligence" means a programme which mimics human cognition. At least some of the things we associate with other minds, such as learning and problem solving can be d...


--- Chunk 2 | Source: Machine learning ---
Content preview (743 chars):


Machine learning gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It is a subfield of computer science. The idea came from work in artificial intelligence. Machine learning explores the study and construction of algorithms which can learn and make predictions on data. Such algorithms follow programmed instructions, but can also make predictions or decisions based on data. They build a model from sample inputs. Machine learning is done where designin...


--- Chunk 3 | Source: Artificial neural network ---
Content preview (979 chars):


A neural network (also called an ANN or an artificial neural network) is a sort of computer software, inspired by biological neurons. Biological brains are capable of solving difficult problems, but each neuron is only responsible for solving a very small part of the problem. Similarly, a neural network is made up of cells that work together to produce a desired result, although each individual cell is only responsible for solving a small part of the problem. This is one method for creating arti...


--- Chunk 4 | Source: Deep learning ---
Content preview (1,053 chars):


Deep learning (also called deep structured learning or hierarchical learning) is a kind of machine learning, which is mostly used with certain kinds of neural networks. As with other kinds of machine-learning, learning sessions can be unsupervised, semi-supervised, or supervised. In many cases, structures are organised so that there is at least one intermediate layer (or hidden layer), between the input layer and the output layer. Certain tasks, such as as recognizing and understanding speech, i...


--- Chunk 5 | Source: Supervised learning ---
Content preview (439 chars):


In machine learning, supervised learning is the task of inferring a function from labelled training data. The results of the training are known beforehand, the system simply learns how to get to these results correctly. Usually, such systems work with vectors. They get the training data and the result of the training as two vectors and produce a "classifier". Usually, the system uses inductive reasoning to generalize the training data....

### 7.4 Compare Retrieval Results Across Strategies

Let's compare what each retriever returns for the same query — checking whether we get different document sources or orderings.

In [ ]:
def compare_retrievers(query, retrievers_dict):
    """Run a query through multiple retrievers and compare results."""
    print(f"Query: \"{query}\"\n")
    for name, retriever in retrievers_dict.items():
        docs = retriever.invoke(query)
        sources = [d.metadata.get('title', d.metadata.get('source', 'PDF'))[:40]
                   for d in docs]
        print(f"  {name:30s} → {sources}")

# Compare all three retrievers
compare_retrievers(
    query="What is the difference between transformers and vision transformers?",
    retrievers_dict={
        "Chroma (Similarity)": similarity_retriever,
        "Chroma (MMR)": mmr_retriever,
        "FAISS (Similarity)": faiss_retriever
    }
)

Query: "What is the difference between transformers and vision transformers?"

  Chroma (Similarity)            → ['', '', '', '', '']
  Chroma (MMR)                   → ['', '', '', '', '']
  FAISS (Similarity)             → ['', '', '', '', '']


## 8. Building the RAG Pipeline with LCEL

Now we assemble the complete RAG pipeline using **LangChain Expression Language (LCEL)**. The pipe operator (`|`) chains components declaratively:

```
User Query → [Retriever → Format Docs] + [Passthrough Query] → Prompt Template → LLM → Answer
```

The LCEL chain has three parallel/sequential stages:
1. **Retrieval:** The query hits the vector store retriever → top-K chunks are returned and formatted as a single context string
2. **Prompt Assembly:** The context + original question are injected into a structured prompt template
3. **Generation:** The LLM (GPT-4.1-mini) generates an answer grounded in the retrieved context

> **Business Scenario:** This is exactly how an HDFC Bank customer support chatbot would work — a customer asks about fixed deposit penalty charges, the retriever finds the relevant policy paragraphs, and the LLM generates a clear, contextual answer.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# RAG prompt — instructs the LLM to answer ONLY from retrieved context
rag_prompt = """You are an assistant who is an expert in question-answering tasks.
Answer the following question using only the following pieces of retrieved context.
If the answer is not in the context, do not make up answers, just say that you don't know.
Keep the answer detailed and well formatted based on the information from the context.

Question:
{question}

Context:
{context}

Answer:
"""

rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)
print("✅ RAG prompt template created")
print(f"   Input variables: {rag_prompt_template.input_variables}")

✅ RAG prompt template created
   Input variables: ['context', 'question']


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# Initialize the LLM — gpt-4.1-mini is the smartest non-reasoning model
chatgpt = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0)

def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using LCEL pipe operator
# Flow: query → {retriever→format | passthrough} → prompt → LLM
qa_rag_chain = (
    {
        "context": similarity_retriever | format_docs,  # retrieve + format
        "question": RunnablePassthrough()                # pass query through
    }
    | rag_prompt_template   # inject into prompt
    | chatgpt               # generate answer
)

print("✅ RAG chain assembled using LCEL")
print("   Retriever : Chroma (similarity, top-5)")
print("   LLM       : gpt-4.1-mini (temperature=0)")

✅ RAG chain assembled using LCEL
   Retriever : Chroma (similarity, top-5)
   LLM       : gpt-4.1-mini (temperature=0)


## 9. Testing the RAG System — Query the Knowledge Base

Let's test the RAG pipeline with a variety of queries to see how well it retrieves and generates answers from our indexed corpus. We'll start with general AI/ML questions and then test edge cases.

> **Key behavior to observe:** When the answer IS in the corpus, the LLM should produce a detailed, grounded response. When the answer is NOT in the corpus, it should say *"I don't know"* — not hallucinate.

In [ ]:
from IPython.display import display, Markdown

# --- Query 1: Broad concept question ---
query = "What is machine learning?"
result = qa_rag_chain.invoke(query)
print(f"Q: {query}")
print("-" * 60)
display(Markdown(result.content))

Q: What is machine learning?
------------------------------------------------------------


Machine learning is a subfield of computer science that gives computers the ability to learn without being explicitly programmed. The concept was introduced by Arthur Samuel in 1959. It originated from work in artificial intelligence and focuses on the study and construction of algorithms that can learn from data and make predictions or decisions based on that data.

Key aspects of machine learning include:

- **Learning from Data:** Machine learning algorithms build models from sample inputs (training data) and use these models to make predictions or decisions.
- **Algorithmic Approach:** While these algorithms follow programmed instructions, they also adapt by learning patterns from data rather than relying solely on explicit programming.
- **Applications:** Machine learning is particularly useful in situations where designing explicit algorithms is difficult or impossible. Examples include spam filtering, network intrusion detection, optical character recognition (OCR), search engines, and computer vision.

There are different types of machine learning, such as:

- **Supervised Learning:** This involves inferring a function from labeled training data, where the desired results are known beforehand. The system learns to map inputs to outputs correctly, often producing a classifier.
- **Deep Learning:** A specialized kind of machine learning that uses multi-layered neural networks (with hidden layers) to process data. Deep learning is especially effective for complex tasks like speech recognition, image understanding, and handwriting recognition. It is inspired by biological nervous systems but differs significantly from actual brain structures.

In summary, machine learning enables computers to learn from data and improve their performance on tasks without being explicitly programmed for each specific task.

In [ ]:
# --- Query 2: Comparison question ---
query = "What is the difference between AI, ML and DL?"
result = qa_rag_chain.invoke(query)
print(f"Q: {query}")
print("-" * 60)
display(Markdown(result.content))

Q: What is the difference between AI, ML and DL?
------------------------------------------------------------


The difference between Artificial Intelligence (AI), Machine Learning (ML), and Deep Learning (DL) can be understood as follows, based on the provided context:

---

### Artificial Intelligence (AI)
- **Definition:** AI is the broadest concept, referring to the ability of a computer program or machine to think, learn, and perform tasks that typically require human intelligence.
- **Purpose:** It aims to create "smart" machines that can work autonomously without being explicitly programmed for every task.
- **Scope:** AI encompasses any system that can interpret external data, learn from it, and adapt flexibly to achieve specific goals or tasks.
- **Examples:** Early AI included tasks like problem-solving and learning. However, as technology advances, some tasks once considered AI (e.g., optical character recognition) are now seen as routine technologies.
- **Historical Note:** The term "Artificial Intelligence" was coined by John McCarthy in 1955.

---

### Machine Learning (ML)
- **Definition:** ML is a subfield of AI focused on giving computers the ability to learn from data without being explicitly programmed for specific tasks.
- **Function:** It involves designing algorithms that build models from sample inputs to make predictions or decisions.
- **Use Cases:** ML is used where explicit programming is difficult, such as spam filtering, network intrusion detection, OCR, search engines, and computer vision.
- **Relation to AI:** ML originated from AI research and is one way to achieve AI by enabling machines to learn from data.
- **Key Point:** ML algorithms improve their performance as they are exposed to more data.

---

### Deep Learning (DL)
- **Definition:** DL is a specialized subset of machine learning that uses multi-layered neural networks to model and understand complex patterns in data.
- **Structure:** DL models have multiple layers (often more than two), including input, hidden (intermediate), and output layers, allowing the system to process information in increasingly abstract ways.
- **Inspiration:** DL is inspired by biological nervous systems but differs significantly from actual brain structures.
- **Capabilities:** DL excels at tasks that are easy for humans but difficult for traditional computers, such as recognizing speech, images, or handwriting.
- **Training:** Deep learning models require large amounts of data (millions or billions of examples) to perform well.
- **Relation to ML:** DL is a type of machine learning focused on hierarchical learning through neural networks.

---

### Summary of Differences
| Aspect               | Artificial Intelligence (AI)                          | Machine Learning (ML)                                  | Deep Learning (DL)                                    |
|----------------------|------------------------------------------------------|-------------------------------------------------------|------------------------------------------------------|
| **Scope**            | Broad field aiming to create intelligent machines    | Subfield of AI focused on learning from data          | Subfield of ML using multi-layer neural networks     |
| **Approach**         | Mimics human cognition and decision-making           | Uses algorithms to build predictive models            | Uses deep neural networks with multiple layers       |
| **Complexity**       | Includes all intelligent behavior by machines        | Focuses on data-driven learning without explicit rules| Focuses on hierarchical feature extraction and abstraction |
| **Examples**         | General AI systems, problem-solving                   | Spam filters, OCR, network intrusion detection         | Speech recognition, image recognition, handwriting recognition |
| **Data Requirements**| Varies                                                | Moderate                                              | Very large datasets needed                            |

---

In essence, **AI** is the overarching concept of machines exhibiting intelligence, **ML** is a method within AI that enables machines to learn from data, and **DL** is a more advanced form of ML that uses deep neural networks to handle complex tasks.

In [ ]:
# --- Query 3: Specific architecture question ---
query = "What is a CNN and how does ResNet improve upon it?"
result = qa_rag_chain.invoke(query)
print(f"Q: {query}")
print("-" * 60)
display(Markdown(result.content))

Q: What is a CNN and how does ResNet improve upon it?
------------------------------------------------------------


**What is a CNN?**

A Convolutional Neural Network (CNN) is a specialized type of Artificial Neural Network (ANN) primarily designed for image-driven pattern recognition tasks. Unlike traditional ANNs, CNNs are structured to efficiently process data with a grid-like topology, such as images, by leveraging spatial hierarchies in the data.

Key characteristics of CNNs include:

- **Layer Types:** CNNs are composed of three main types of layers:
  1. **Convolutional layers:** These layers apply convolution operations to the input, where neurons are connected only to local regions of the input volume. This local connectivity allows the network to capture spatial features effectively.
  2. **Pooling layers:** These reduce the spatial dimensions (height and width) of the data, helping to decrease computational complexity and control overfitting.
  3. **Fully-connected layers:** These layers connect every neuron in one layer to every neuron in the next, typically used towards the end of the network for classification.

- **Input and Output Dimensions:** The input to a CNN is typically a 3D volume (height × width × depth), where depth corresponds to the number of channels (e.g., 3 for RGB images). The output is often a condensed volume representing class scores.

- **Stacking Convolutional Layers:** It is common to stack multiple convolutional layers before each pooling layer. This stacking allows the network to learn more complex and abstract features by increasing the receptive field of neurons in deeper layers (e.g., stacking three 3×3 convolutional layers results in a 7×7 receptive field).

- **Use of Small Filters:** CNNs often use small convolutional filters (e.g., 3×3) with stride one and zero-padding to preserve spatial dimensions, which helps reduce computational complexity while maintaining the ability to learn detailed features.

- **Activation Functions:** Rectified Linear Units (ReLU) are commonly used after convolutional layers to introduce non-linearity.

- **Challenges:** CNNs can be resource-heavy, especially with large input images and many filters, leading to high memory usage and computational demands.

---

**How does ResNet improve upon CNNs?**

While the provided context does not explicitly detail the architecture of ResNet, it references the use of ResNet-101 in the context of image classification and localization tasks, highlighting its superior performance:

- **Performance Gains:** ResNet-101 achieves a top-5 classification error of 4.6% and a top-5 localization error of 10.6% on validation sets, significantly outperforming previous models and winning the 1st place in the ImageNet localization task in ILSVRC 2015.

- **Error Reduction:** The use of ResNet leads to a 64% relative reduction in error compared to earlier results, indicating a substantial improvement in accuracy.

- **Integration with Detection Frameworks:** ResNet is used effectively within frameworks like Faster R-CNN for object detection, showing its versatility and robustness.

Although the context does not provide the architectural details of ResNet, it is known from the literature that ResNet (Residual Network) improves upon traditional CNNs by introducing **residual connections** or **skip connections**. These connections help mitigate the problem of vanishing gradients in very deep networks, allowing the training of substantially deeper models without degradation in performance. This leads to better feature learning and improved accuracy.

---

### Summary

- **CNNs** are deep learning models designed for image data, using convolutional, pooling, and fully-connected layers to extract and classify features from images efficiently.
- **ResNet** enhances CNNs by enabling much deeper networks through residual connections, which improve training stability and accuracy, as evidenced by its superior performance in large-scale image classification and localization tasks.

In [ ]:
# --- Query 4: Transformer-specific question ---
query = "How is self-attention important in transformers?"
result = qa_rag_chain.invoke(query)
print(f"Q: {query}")
print("-" * 60)
display(Markdown(result.content))

Q: How is self-attention important in transformers?
------------------------------------------------------------


Self-attention is a fundamental component in transformers and plays a crucial role in their effectiveness. Based on the provided context, its importance in transformers can be detailed as follows:

### 1. Core Mechanism in Transformer Architecture
- The Transformer architecture relies entirely on **self-attention** to compute representations of its input and output sequences, without using sequence-aligned recurrent neural networks (RNNs) or convolutional layers.
- Both the encoder and decoder stacks in a Transformer include multi-head self-attention mechanisms as their first sub-layer in each layer. This allows the model to process input sequences by relating different positions of the sequence to each other.

### 2. Global Integration of Information
- Self-attention enables the model to **integrate information globally** across the entire input. For example, in Vision Transformers (ViT), some attention heads attend to most of the image even in the lowest layers, demonstrating the model’s ability to capture long-range dependencies and global context.
- This global attention contrasts with convolutional neural networks (CNNs), which typically have localized receptive fields. Self-attention allows the model to flexibly attend to semantically relevant regions regardless of their spatial distance.

### 3. Flexibility in Attention Scope
- Different attention heads can focus on different ranges: some attend globally, while others maintain localized attention. This diversity allows the model to capture both fine-grained local features and broad contextual information.
- The attention distance (analogous to receptive field size in CNNs) tends to increase with network depth, meaning deeper layers integrate information over larger parts of the input.

### 4. Advantages Over Traditional Architectures
- Unlike CNNs, which have built-in inductive biases such as translation equivariance and locality, transformers with self-attention do not rely on these biases. This makes them more flexible and capable of generalizing better when trained on sufficiently large datasets.
- Self-attention allows transformers to scale well and achieve state-of-the-art performance on various tasks, especially when combined with large-scale self-supervised pre-training.

### 5. Multi-Head Self-Attention
- The use of **multi-head self-attention** allows the model to jointly attend to information from different representation subspaces at different positions, enhancing the richness of the learned representations.

### Summary
Self-attention is important in transformers because it:
- Enables the model to relate all parts of the input sequence to each other, capturing global dependencies.
- Provides flexibility to attend both locally and globally, adapting to the needs of the task.
- Replaces traditional sequence models like RNNs and CNNs by offering a scalable and effective mechanism for sequence representation.
- Supports multi-head attention, which enriches the model’s ability to capture diverse features.
- Facilitates superior performance on large-scale tasks, especially when combined with self-supervised pre-training.

Thus, self-attention is the key mechanism that underpins the transformer's ability to process sequences effectively and achieve state-of-the-art results in natural language processing and computer vision tasks.

In [ ]:
# --- Query 5: Comparison within corpus ---
query = "What is the difference between transformers and vision transformers?"
result = qa_rag_chain.invoke(query)
print(f"Q: {query}")
print("-" * 60)
display(Markdown(result.content))

Q: What is the difference between transformers and vision transformers?
------------------------------------------------------------


The difference between **Transformers** and **Vision Transformers (ViT)** lies primarily in their application domain, input representation, and architectural adaptations to handle image data effectively. Below is a detailed explanation based on the provided context:

---

### 1. **Original Transformers (NLP Context)**
- **Purpose:** Designed primarily for natural language processing (NLP) tasks such as machine translation.
- **Input:** Sequences of tokens (words or subwords) from text.
- **Architecture:** Uses self-attention mechanisms to model relationships between tokens in a sequence, capturing long-range dependencies.
- **Inductive Bias:** Transformers in NLP do not have explicit spatial or locality biases because language is inherently sequential rather than spatial.

---

### 2. **Vision Transformers (ViT)**
- **Purpose:** Adaptation of the Transformer architecture for image recognition tasks.
- **Input Representation:**
  - Instead of processing raw pixels or convolutional feature maps, ViT splits an image into fixed-size **patches** (e.g., 16×16 pixels).
  - Each patch is **flattened** and linearly projected into a lower-dimensional embedding space, treating patches as tokens analogous to words in NLP.
  - A **learned position embedding** is added to each patch embedding to encode spatial information, since the Transformer itself does not inherently understand 2D spatial structure.
- **Architecture:**
  - The core Transformer architecture (multi-head self-attention and MLP layers) remains largely unchanged.
  - Self-attention layers in ViT are **global**, allowing integration of information across the entire image from the earliest layers.
  - Unlike CNNs, ViT has **much less image-specific inductive bias**: it does not bake in locality, translation equivariance, or 2D neighborhood structure except through the initial patching and position embeddings.
- **Training:**
  - ViT requires **large-scale pre-training** on massive datasets (e.g., JFT-300M) to perform competitively, as it lacks the strong inductive biases of CNNs.
  - When pre-trained at scale, ViT can outperform or match state-of-the-art convolutional networks on image classification benchmarks.
- **Advantages:**
  - ViT can integrate global context early and efficiently.
  - It scales well with model size and dataset size.
  - It requires fewer computational resources to reach comparable performance than some CNNs.

---

### 3. **Key Differences Summarized**

| Aspect                     | Transformer (NLP)                          | Vision Transformer (ViT)                          |
|----------------------------|------------------------------------------|--------------------------------------------------|
| **Input Type**             | Sequence of text tokens                   | Sequence of image patches (flattened pixel blocks) |
| **Spatial Structure**      | Sequential, 1D tokens                     | 2D image patches with learned position embeddings |
| **Inductive Bias**         | None specific to spatial locality        | Minimal; only patching and position embeddings encode 2D structure |
| **Self-Attention Scope**   | Global over tokens                        | Global over image patches                          |
| **Training Data Requirements** | Large text corpora                      | Very large image datasets needed for good performance |
| **Performance**            | State-of-the-art in NLP                   | Competitive with or better than CNNs when pre-trained at scale |
| **Use of Convolutions**    | None                                     | None in pure ViT; hybrids combine CNN features with ViT |

---

### 4. **Additional Notes**
- ViT can be fine-tuned at higher image resolutions by interpolating position embeddings.
- Hybrid models exist that combine CNN feature maps with Transformer layers, but pure ViT models operate directly on image patches.
- ViT’s global self-attention allows it to integrate information across the entire image even in early layers, unlike CNNs which rely on local receptive fields.

---

### **In summary:**

**Transformers** are general sequence models originally designed for text, processing sequences of tokens with self-attention but lacking spatial inductive biases. **Vision Transformers** adapt this architecture to images by splitting images into patches treated as tokens, adding learned position embeddings to encode spatial information, and applying global self-attention to these patches. This approach removes the need for convolutional inductive biases, relying instead on large-scale training to achieve competitive or superior performance in image recognition tasks.

In [ ]:
# --- Query 6: NLP domain question ---
query = "What is NLP and its relation to linguistics?"
result = qa_rag_chain.invoke(query)
print(f"Q: {query}")
print("-" * 60)
display(Markdown(result.content))

Q: What is NLP and its relation to linguistics?
------------------------------------------------------------


**What is NLP and its relation to linguistics?**

**Natural Language Processing (NLP)** is a field within Artificial Intelligence (AI) that focuses on enabling computers to automatically understand, interpret, and generate human languages. The term "Natural Language" specifically refers to human languages, distinguishing it from computer or programming languages. The primary goal of NLP is twofold:

1. To program computers to automatically understand human languages.
2. To enable computers to automatically write or speak in human languages.

NLP is closely related to **linguistics**, the scientific study of language, because understanding and generating human language requires knowledge of linguistic structures such as syntax, semantics, and pragmatics. Linguistics provides the theoretical foundation and insights into how human languages work, which NLP leverages to build computational models that can process language effectively.

---

**Note:** The context also mentions another discipline abbreviated as "NLP" — **Neurolinguistic Programming** — which is unrelated to Natural Language Processing. Neurolinguistic Programming is a communication method developed in the 1970s, based on the idea that neurological processes, language, and behavior are linked, and that changing behavior can help achieve life goals. However, this form of NLP is considered unsupported by current scientific evidence and is distinct from Natural Language Processing.

---

**Summary:**

- **Natural Language Processing (NLP)** is an AI field aimed at programming computers to understand and generate human language.
- It is related to **linguistics** because it relies on linguistic knowledge to model and process natural languages.
- "Natural Language" means human language, not computer languages.
- NLP should not be confused with Neurolinguistic Programming, which is a separate, non-scientific communication method.

### 9.1 Testing the Hallucination Guard

These queries ask about topics that are **NOT** in our corpus. A well-designed RAG system should refuse to answer rather than hallucinate.

> **Why this matters:** In a banking chatbot, hallucinated interest rates or incorrect policy information could lead to regulatory violations. The system MUST say "I don't know" when the answer isn't in the knowledge base.

In [ ]:
# --- Query 7: Topic NOT in corpus (should say "I don't know") ---
query = "What is LangGraph?"
result = qa_rag_chain.invoke(query)
print(f"Q: {query}")
print("-" * 60)
display(Markdown(result.content))

Q: What is LangGraph?
------------------------------------------------------------


The provided context does not contain any information about "LangGraph." Therefore, I do not know what LangGraph is based on the given information.

In [ ]:
# --- Query 8: Another out-of-scope query ---
query = "What is an Agentic AI System?"
result = qa_rag_chain.invoke(query)
print(f"Q: {query}")
print("-" * 60)
display(Markdown(result.content))

Q: What is an Agentic AI System?
------------------------------------------------------------


An **Agentic AI System** can be understood as an ideal intelligent machine described as a **flexible agent** that perceives its environment and takes actions to maximize its chance of success at some goal or objective. This concept is derived from the definition of artificial intelligence as a system’s ability to correctly interpret external data, learn from such data, and use those learnings to achieve specific goals and tasks through flexible adaptation.

In other words, an agentic AI system acts autonomously by sensing its surroundings, making decisions, and performing actions aimed at achieving predefined objectives, adapting its behavior based on the information it gathers. This aligns with the idea of an AI agent that operates independently rather than following rigid, pre-encoded commands.

The context does not provide a direct, explicit definition of "Agentic AI System," but based on the description of an ideal intelligent machine as a flexible agent, this is the closest and most detailed explanation available.

## 10. RAG Pipeline with MMR Retriever

Let's build a second RAG chain that uses the **MMR retriever** instead of plain similarity search. This can improve answer quality by providing more diverse context to the LLM.

In [ ]:
# Build a second RAG chain with MMR retrieval
qa_rag_chain_mmr = (
    {
        "context": mmr_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt_template
    | chatgpt
)

# Compare: same query through both chains
query = "How does a ResNet work and why is it better than a standard CNN?"

print("=" * 70)
print("SIMILARITY SEARCH RAG")
print("=" * 70)
result_sim = qa_rag_chain.invoke(query)
display(Markdown(result_sim.content))

print("\n" + "=" * 70)
print("MMR RAG")
print("=" * 70)
result_mmr = qa_rag_chain_mmr.invoke(query)
display(Markdown(result_mmr.content))

SIMILARITY SEARCH RAG


**How does a ResNet work and why is it better than a standard CNN?**

---

### How ResNet Works

ResNet (Residual Network) introduces a novel architecture designed to ease the training of very deep neural networks by addressing the degradation problem observed in standard deep CNNs.

- **Residual Learning Framework:**  
  Instead of directly trying to learn a desired underlying mapping \( H(x) \) through stacked layers, ResNet reformulates the problem to learn a residual mapping \( F(x) := H(x) - x \). This means the network learns the difference between the input \( x \) and the desired output \( H(x) \), and the original mapping is expressed as:  
  \[
  H(x) = F(x) + x
  \]

- **Shortcut Connections (Identity Mappings):**  
  This residual function \( F(x) \) is implemented using shortcut connections that skip one or more layers and perform identity mapping. The output of these shortcut connections is added element-wise to the output of the stacked layers (Fig. 2 in the context). These shortcuts:  
  - Add no extra parameters or computational complexity.  
  - Allow the network to propagate information and gradients more directly through the network.

- **Building Blocks:**  
  ResNet uses building blocks consisting of a few convolutional layers (e.g., 3-layer bottleneck blocks in deeper ResNets like 50, 101, and 152 layers). The shortcut connections are added to these blocks, enabling very deep networks to be trained effectively.

- **Training Details:**  
  ResNets are trained end-to-end using standard stochastic gradient descent with backpropagation. Batch Normalization (BN) is used to maintain healthy gradient flow and signal variance, and no dropout is necessary.

---

### Why ResNet is Better than Standard CNNs

- **Solves the Degradation Problem:**  
  In standard deep CNNs (referred to as "plain" networks), increasing depth beyond a certain point leads to higher training error and worse performance, a phenomenon called the degradation problem. This is not due to vanishing gradients (which BN helps prevent), but likely due to optimization difficulties where deeper plain networks converge very slowly or get stuck in poor local minima.

- **Easier Optimization:**  
  By learning residual functions \( F(x) \) instead of direct mappings \( H(x) \), ResNets make it easier for the network to optimize. If the identity mapping is optimal, the residual function can simply push the residual to zero, which is easier than fitting an identity mapping with multiple nonlinear layers.

- **Enables Very Deep Networks:**  
  ResNets allow training of extremely deep networks (e.g., 50, 101, 152 layers, and even over 1000 layers in experiments) without suffering from degradation. These deep residual networks achieve significantly lower training and validation errors compared to their plain counterparts of the same depth.

- **Improved Accuracy:**  
  Experiments on ImageNet and CIFAR-10 datasets show that ResNets outperform plain networks and previous state-of-the-art models. For example:  
  - A 34-layer ResNet outperforms an 18-layer ResNet by a large margin, reversing the trend seen in plain networks.  
  - The 152-layer ResNet achieves a top-5 validation error of 4.49% on ImageNet, outperforming all previous single-model results and even ensembles of other models.  
  - On CIFAR-10, deeper ResNets consistently reduce classification error compared to plain networks.

- **No Extra Parameters or Complexity:**  
  The identity shortcut connections add no extra parameters or computational cost, making ResNets efficient despite their depth.

- **Generalization and Versatility:**  
  The residual learning principle is generic and has been successfully applied beyond image classification, including detection and segmentation tasks, winning multiple competitions (e.g., ILSVRC 2015).

---

### Summary

| Aspect                     | Standard CNN (Plain Network)                  | ResNet (Residual Network)                          |
|----------------------------|-----------------------------------------------|---------------------------------------------------|
| Learning Objective          | Directly fit \( H(x) \)                        | Fit residual \( F(x) = H(x) - x \)                |
| Architecture                | Stacked convolutional layers                   | Stacked layers + identity shortcut connections    |
| Optimization Difficulty     | High for very deep networks (degradation)     | Easier due to residual learning                    |
| Training Error with Depth   | Increases beyond certain depth                 | Decreases or remains low with increased depth     |
| Model Depth                 | Limited by optimization issues                 | Can be very deep (50, 101, 152+ layers)           |
| Parameters & Complexity     | Standard                                       | No extra parameters or complexity from shortcuts  |
| Performance (e.g., ImageNet)| Lower accuracy, higher error                    | State-of-the-art accuracy, lower error             |

---

**In conclusion, ResNet works by reformulating the learning task to residual functions with shortcut connections, which makes training very deep networks feasible and effective. This leads to better optimization, higher accuracy, and the ability to build much deeper models than standard CNNs.**


MMR RAG


### How does a ResNet work and why is it better than a standard CNN?

#### How ResNet Works:
ResNet, or Residual Network, is a type of deep convolutional neural network architecture designed to enable the training of very deep networks without suffering from the degradation problem (where deeper networks perform worse than shallower ones).

- **Residual Blocks:**  
  The core idea of ResNet is the use of *residual blocks* that include *shortcut connections* (also called skip connections). Instead of learning a direct mapping from input to output, each residual block learns a residual function with reference to the input. Formally, if the desired underlying mapping is \( H(x) \), the residual block learns \( F(x) = H(x) - x \), so the output is \( F(x) + x \). This is implemented by adding the input \( x \) to the output of a few stacked convolutional layers.

- **Shortcut Connections:**  
  These connections bypass one or more layers by performing identity mapping, and their outputs are added to the outputs of the stacked layers. This helps in mitigating the vanishing gradient problem by allowing gradients to flow directly through the network during backpropagation.

- **Bottleneck Blocks:**  
  For very deep ResNets (e.g., 50, 101, 152 layers), a 3-layer bottleneck block is used, which consists of a 1×1 convolution to reduce dimensions, a 3×3 convolution, and another 1×1 convolution to restore dimensions. This design reduces computational complexity while maintaining representational power.

- **Depth and Architecture:**  
  ResNets have been successfully scaled to very deep architectures such as 50, 101, and 152 layers. For example, a 152-layer ResNet has 11.3 billion FLOPs, which is still less than the computational complexity of VGG-16/19 networks (15.3/19.6 billion FLOPs).

- **Training Details:**  
  On datasets like CIFAR-10, ResNets use identity shortcuts and maintain the same depth, width, and number of parameters as their plain (non-residual) counterparts, but achieve better performance. They are trained with batch normalization and weight initialization techniques, without dropout.

#### Why ResNet is Better than Standard CNNs:

- **Avoids Degradation Problem:**  
  Standard CNNs suffer from the degradation problem where increasing depth beyond a certain point leads to higher training error and worse performance. ResNets do not exhibit this problem, allowing much deeper networks to be trained effectively.

- **Improved Accuracy with Depth:**  
  ResNets show significant accuracy improvements as depth increases. For example, 50/101/152-layer ResNets outperform shallower 34-layer ResNets by considerable margins across various evaluation metrics.

- **Lower Computational Complexity:**  
  Despite being deeper, ResNets can have lower or comparable computational complexity compared to other deep networks like VGG, due to the use of bottleneck blocks and efficient architecture design.

- **State-of-the-Art Performance:**  
  ResNets have achieved top results in major benchmarks such as ImageNet and CIFAR-10. For instance, a 152-layer ResNet achieved a top-5 validation error of 4.49% on ImageNet, outperforming previous ensemble methods. Ensembles of ResNets have further reduced error rates, winning the ILSVRC 2015 competition.

- **Better Gradient Flow:**  
  The shortcut connections facilitate better gradient flow during backpropagation, making it easier to train very deep networks without vanishing or exploding gradients.

- **Flexibility and Scalability:**  
  ResNet architectures can be scaled up to hundreds or even thousands of layers (e.g., ResNet-110, ResNet-1202 on CIFAR-10) while maintaining or improving performance, which is difficult for standard CNNs.

---

### Summary:
ResNet works by introducing residual learning through shortcut connections that allow the network to learn residual functions instead of direct mappings. This innovation enables the training of very deep networks without degradation in performance. Compared to standard CNNs, ResNets are better because they avoid the degradation problem, improve accuracy with increased depth, maintain computational efficiency, and achieve state-of-the-art results on challenging image recognition tasks.

## 11. RAG Pipeline with FAISS Retriever

Let's also build a RAG chain backed by FAISS to verify that both vector stores produce comparable results.

In [ ]:
# Build RAG chain with FAISS backend
qa_rag_chain_faiss = (
    {
        "context": faiss_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt_template
    | chatgpt
)

# Test with a query
query = "What is word sense disambiguation?"
result = qa_rag_chain_faiss.invoke(query)
print(f"Q: {query} (FAISS-backed RAG)")
print("-" * 60)
display(Markdown(result.content))

Q: What is word sense disambiguation? (FAISS-backed RAG)
------------------------------------------------------------


The provided context does not contain a direct definition or explanation of "word sense disambiguation." Therefore, based on the given information, I do not know what word sense disambiguation is.

## 12. RAG with Source Attribution

In production systems, users need to know **where** the answer came from — which documents were retrieved. This is critical for trust, auditability, and regulatory compliance.

> **Business Scenario:** When an insurance claims adjuster asks the RAG system about coverage terms, the response must cite the exact policy document and section — not just provide an unsourced answer.

We build a helper that returns both the answer AND the source documents.

In [ ]:
def rag_with_sources(query, retriever, prompt_template, llm):
    """Run RAG and return both the answer and source documents."""
    # Step 1: Retrieve relevant documents
    retrieved_docs = retriever.invoke(query)

    # Step 2: Format context
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # Step 3: Build the prompt and invoke LLM
    chain = prompt_template | llm
    result = chain.invoke({"question": query, "context": context})

    return {
        "query": query,
        "answer": result.content,
        "sources": [
            {
                "title": d.metadata.get('title', d.metadata.get('source', 'Unknown')),
                "chars": len(d.page_content)
            }
            for d in retrieved_docs
        ]
    }

# Test with source attribution
response = rag_with_sources(
    query="How is self-attention important in transformers?",
    retriever=similarity_retriever,
    prompt_template=rag_prompt_template,
    llm=chatgpt
)

print(f"Q: {response['query']}\n")
display(Markdown(response['answer']))

print("\n📚 Sources Used:")
for i, src in enumerate(response['sources'], 1):
    print(f"   {i}. {src['title']} ({src['chars']:,} chars)")

Q: How is self-attention important in transformers?



Self-attention is a fundamental component in transformers and plays a crucial role in their architecture and performance. Based on the provided context, the importance of self-attention in transformers can be detailed as follows:

### 1. Core Mechanism in Transformer Architecture
- **Self-attention is the primary mechanism** by which transformers compute representations of their input and output sequences. Unlike previous models that rely on recurrent neural networks (RNNs) or convolutional neural networks (CNNs), transformers use self-attention exclusively to capture dependencies within the input data.
- In the **encoder stack**, each layer contains a multi-head self-attention sub-layer followed by a position-wise fully connected feed-forward network. Residual connections and layer normalization are applied around these sub-layers to stabilize training and improve gradient flow.
- In the **decoder stack**, self-attention is also used but with a masking mechanism to prevent positions from attending to subsequent positions, ensuring autoregressive generation.

### 2. Global Integration of Information
- Self-attention allows the model to **integrate information globally across the entire input**. For example, in Vision Transformers (ViT), some attention heads in the lowest layers attend to most of the image, demonstrating the model’s ability to capture long-range dependencies and global context early in the processing pipeline.
- This global attention contrasts with the localized receptive fields of CNNs and enables transformers to attend to semantically relevant regions regardless of their spatial distance.

### 3. Flexibility in Attention Scope
- Different attention heads can focus on different ranges: some attend globally, while others maintain **highly localized attention**. This diversity allows the model to capture both fine-grained local features and broad contextual information.
- The **attention distance increases with network depth**, meaning deeper layers tend to integrate information over larger spatial extents.

### 4. Advantages Over Traditional Architectures
- Transformers, through self-attention, do not rely on inductive biases inherent to CNNs such as translation equivariance and locality. This makes them more flexible and capable of generalizing better when trained on sufficiently large datasets.
- Large-scale self-supervised pre-training combined with self-attention enables transformers to achieve or surpass state-of-the-art results on various tasks, including image classification benchmarks.

### 5. Scalability and Efficiency Considerations
- Naively applying self-attention to images is computationally expensive due to quadratic complexity with respect to input size. However, various approximations and architectural choices (e.g., patch-based processing in ViT) make self-attention scalable and practical for large inputs.
- Self-attention’s ability to replace convolutions or augment CNNs has been explored, showing promising results in vision and other domains.

---

### Summary
Self-attention is important in transformers because it:
- Enables the model to compute contextualized representations by attending to all parts of the input sequence simultaneously.
- Facilitates global integration of information, allowing the model to capture long-range dependencies effectively.
- Provides flexibility for different attention heads to focus on local or global features.
- Supports scalability and strong performance when combined with large-scale pre-training.
- Serves as the foundational mechanism that distinguishes transformers from RNNs and CNNs, enabling superior performance in NLP and vision tasks.

This comprehensive role of self-attention underpins the success of transformers across a wide range of applications.


📚 Sources Used:
   1.  (870 chars)
   2.  (1,826 chars)
   3.  (911 chars)
   4.  (3,457 chars)
   5.  (3,489 chars)


## 13. Similarity Search with Relevance Scores

Understanding the **similarity scores** helps debug retrieval quality. Higher scores (closer to 1.0 for cosine) mean the chunk is more relevant to the query.

> **Tip:** In production, you'd set a minimum score threshold (e.g., 0.7) to filter out low-relevance chunks before passing them to the LLM — reducing noise in the context.

In [ ]:
# Similarity search with scores — see how relevant each chunk is
query = "What is the difference between transformers and vision transformers?"
results_with_scores = chroma_db.similarity_search_with_relevance_scores(query, k=5)

print(f"Query: \"{query}\"\n")
print(f"{'Rank':<6} {'Score':<10} {'Source':<45} {'Chars'}")
print("-" * 80)
for i, (doc, score) in enumerate(results_with_scores, 1):
    title = doc.metadata.get('title', doc.metadata.get('source', 'PDF'))[:42]
    print(f"{i:<6} {score:<10.4f} {title:<45} {len(doc.page_content):,}")

Query: "What is the difference between transformers and vision transformers?"

Rank   Score      Source                                        Chars
--------------------------------------------------------------------------------
1      0.5392                                                   3,457
2      0.5314                                                   3,500
3      0.4980                                                   3,469
4      0.4980                                                   3,489
5      0.4793                                                   3,435


## 14. Cost & Performance Estimates

In [ ]:
# Estimate costs for this lab run
cost_estimates = {
    "Embedding (indexing ~200 chunks)": {
        "estimate_usd": 0.005,
        "note": "~200 chunks × ~500 tokens avg × $0.02/1M tokens"
    },
    "Embedding (queries, ~15 queries)": {
        "estimate_usd": 0.001,
        "note": "~15 queries × ~20 tokens avg"
    },
    "LLM (gpt-4.1-mini, ~12 RAG calls)": {
        "estimate_usd": 0.012,
        "note": "~12 calls × ~2000 input + 500 output tokens"
    },
}

print("💰 ESTIMATED COST PER RUN")
print("=" * 65)
total = 0
for component, info in cost_estimates.items():
    print(f"  {component:<42s} ${info['estimate_usd']:.3f}  ({info['note']})")
    total += info["estimate_usd"]
print("-" * 65)
print(f"  {'TOTAL':<42s} ${total:.3f}")
print("\n💡 Total cost is well under $0.05 per complete lab run.")

💰 ESTIMATED COST PER RUN
  Embedding (indexing ~200 chunks)           $0.005  (~200 chunks × ~500 tokens avg × $0.02/1M tokens)
  Embedding (queries, ~15 queries)           $0.001  (~15 queries × ~20 tokens avg)
  LLM (gpt-4.1-mini, ~12 RAG calls)          $0.012  (~12 calls × ~2000 input + 500 output tokens)
-----------------------------------------------------------------
  TOTAL                                      $0.018

💡 Total cost is well under $0.05 per complete lab run.


---

## 15. Conclusion & Next Steps

### What We Covered

| Step | Key Takeaway |
|------|-------------|
| **Document Loading** | LangChain loaders handle JSON, JSONL, and PDF sources with metadata extraction |
| **Text Chunking** | `RecursiveCharacterTextSplitter` with 3500-char chunks and 200-char overlap balances granularity and context preservation |
| **Chroma Vector Store** | Lightweight, persistent vector DB — great for prototyping; supports cosine similarity and metadata filtering |
| **FAISS Vector Store** | High-performance vector index — faster for large-scale retrieval; save/load to disk |
| **Similarity vs. MMR** | MMR adds diversity to retrieved chunks, reducing redundancy when documents overlap |
| **LCEL RAG Pipeline** | The pipe operator (`\|`) composes retriever → formatter → prompt → LLM into a clean, declarative chain |
| **Source Attribution** | Returning source documents alongside answers is critical for trust and auditability |
| **Hallucination Guard** | A well-crafted RAG prompt makes the LLM say "I don't know" when context doesn't contain the answer |

### Try on Your Own

1. **Tune chunk size:** Re-run the pipeline with `chunk_size=1000` and `chunk_size=5000` — observe how retrieval quality and answer detail change
2. **Add metadata filtering:** Use Chroma's `filter` parameter to restrict retrieval to only Wikipedia articles or only PDF papers
3. **Experiment with `fetch_k` in MMR:** Try `fetch_k=50` vs `fetch_k=10` and see how diversity changes
4. **Score threshold filtering:** Modify the retriever to use `search_type="similarity_score_threshold"` with `score_threshold=0.7` to filter out low-relevance chunks
5. **Swap embedding models:** Try `text-embedding-3-large` (3072 dims) and compare retrieval quality against `text-embedding-3-small`
6. **Add your own documents:** Index a PDF of your choice (e.g., an RBI circular or a company policy document) and query it

---